# Movie Dataset — EDA & Preprocessing

This notebook explores the raw movie dataset, identifies data quality issues,
and applies the preprocessing steps needed before generating embeddings.

In [ ]:
# standard data science stack
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# make plots look clean
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1 · Load the raw data

In [ ]:
# read the CSV that ships with the repo
df_raw = pd.read_csv("../data/raw/movies.csv")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()

## 2 · Quick schema overview

In [ ]:
# data types and non-null counts
df_raw.info()

In [ ]:
# basic stats for numeric columns
df_raw.describe()

## 3 · Missing values

In [ ]:
# count and percentage of nulls per column
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct}).sort_values("missing_pct", ascending=False)

In [ ]:
# visualize missingness
fig, ax = plt.subplots(figsize=(10, 4))
missing[missing > 0].sort_values().plot.barh(ax=ax, color="coral")
ax.set_xlabel("Number of missing values")
ax.set_title("Missing Values per Column")
plt.tight_layout()
plt.show()

## 4 · Duplicate analysis

In [ ]:
# check for exact row duplicates
exact_dupes = df_raw.duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")

# check for same-title-same-date duplicates (the meaningful definition)
title_date_dupes = df_raw.duplicated(subset=["names", "date_x"]).sum()
print(f"Same (title + date) duplicates: {title_date_dupes}")

In [ ]:
# peek at a few duplicate groups
dupe_mask = df_raw.duplicated(subset=["names", "date_x"], keep=False)
if dupe_mask.any():
    sample_name = df_raw.loc[dupe_mask, "names"].iloc[0]
    display(df_raw[df_raw["names"] == sample_name])

## 5 · Score distribution

In [ ]:
# histogram of review scores
fig, ax = plt.subplots()
df_raw["score"].dropna().hist(bins=30, ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Score (0-100)")
ax.set_ylabel("Number of Movies")
ax.set_title("Distribution of Movie Scores")
plt.tight_layout()
plt.show()

## 6 · Genre breakdown

In [ ]:
# each row may have comma-separated genres — split and count
all_genres = df_raw["genre"].dropna().str.split(", ").explode()
top_genres = all_genres.value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
top_genres.sort_values().plot.barh(ax=ax, color="mediumpurple")
ax.set_xlabel("Count")
ax.set_title("Top 20 Genres")
plt.tight_layout()
plt.show()

## 7 · Release date & year trends

In [ ]:
# extract year from the date string (format: MM/DD/YYYY)
df_raw["year"] = df_raw["date_x"].str.strip().str.split("/").str[-1].astype(float)

fig, ax = plt.subplots()
df_raw["year"].dropna().astype(int).value_counts().sort_index().plot(ax=ax, color="teal")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Movies")
ax.set_title("Movies Released per Year")
plt.tight_layout()
plt.show()

## 8 · Budget & revenue

In [ ]:
# quick look at financial columns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_raw["budget_x"].dropna().clip(upper=5e8).hist(bins=40, ax=axes[0], color="goldenrod", edgecolor="white")
axes[0].set_title("Budget Distribution (clipped at $500M)")
axes[0].set_xlabel("Budget ($)")

df_raw["revenue"].dropna().clip(upper=2e9).hist(bins=40, ax=axes[1], color="seagreen", edgecolor="white")
axes[1].set_title("Revenue Distribution (clipped at $2B)")
axes[1].set_xlabel("Revenue ($)")

plt.tight_layout()
plt.show()

In [ ]:
# scatter: budget vs revenue (log scale)
fig, ax = plt.subplots(figsize=(8, 6))
mask = (df_raw["budget_x"] > 0) & (df_raw["revenue"] > 0)
ax.scatter(df_raw.loc[mask, "budget_x"], df_raw.loc[mask, "revenue"],
           alpha=0.3, s=10, color="darkorange")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Budget ($, log)")
ax.set_ylabel("Revenue ($, log)")
ax.set_title("Budget vs Revenue")
plt.tight_layout()
plt.show()

## 9 · Language & country

In [ ]:
# top 15 original languages
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_raw["orig_lang"].value_counts().head(15).sort_values().plot.barh(ax=axes[0], color="salmon")
axes[0].set_title("Top 15 Original Languages")

df_raw["country"].value_counts().head(15).sort_values().plot.barh(ax=axes[1], color="cornflowerblue")
axes[1].set_title("Top 15 Countries")

plt.tight_layout()
plt.show()

## 10 · Overview text length

In [ ]:
# word count of the overview field
df_raw["overview_len"] = df_raw["overview"].fillna("").apply(lambda x: len(x.split()))

fig, ax = plt.subplots()
df_raw["overview_len"].hist(bins=50, ax=ax, color="slateblue", edgecolor="white")
ax.set_xlabel("Word Count")
ax.set_ylabel("Frequency")
ax.set_title("Overview Length (in words)")
plt.tight_layout()
plt.show()

print(f"Median overview length: {df_raw['overview_len'].median():.0f} words")
print(f"Max overview length:    {df_raw['overview_len'].max()} words")

## 11 · Correlation heatmap (numeric columns)

In [ ]:
# correlations between numeric features
numeric_cols = ["score", "budget_x", "revenue"]
corr = df_raw[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

---
# 🧹 Preprocessing Pipeline

The steps below clean the data and prepare it for embedding generation.

## 12 · Remove placeholder overviews

In [ ]:
# some movies have a generic placeholder instead of a real overview — drop them
placeholder = "We don't have an overview translated in English. Help us expand our database by adding one."
df = df_raw[df_raw["overview"] != placeholder].copy()
print(f"Dropped {len(df_raw) - len(df):,} placeholder rows  →  {len(df):,} remaining")

## 13 · Fill missing genres

In [ ]:
# fill missing genre with "Unknown" so downstream code doesn't break
df["genre"] = df["genre"].fillna("Unknown")
print("Null genres remaining:", df["genre"].isnull().sum())

## 14 · De-duplicate by (title, date)

In [ ]:
# some movies appear more than once — aggregate them into a single row
before = len(df)
df = df.groupby(["names", "date_x"]).agg({
    "score": "mean",          # average scores
    "genre": "first",         # keep first genre string
    "overview": "first",      # keep first overview
    "crew": "first",          # keep first crew info
    "orig_title": "first",    # keep original title
    "status": "first",        # keep first status
    "orig_lang": "first",     # keep original language
    "budget_x": "sum",        # sum budgets
    "revenue": "sum",         # sum revenues
    "country": "first"        # keep first country
}).reset_index()

print(f"De-duplication: {before:,}  →  {len(df):,} rows")

## 15 · Build the text representation

In [ ]:
# combine all fields into a single string that the embedding model will encode
def textify_movie(row):
    def clean(val):
        if pd.isna(val) or val == "":
            return "Unknown"
        return str(val)

    name     = clean(row.get("names"))
    date     = clean(row.get("date_x"))
    score    = clean(row.get("score"))
    genres   = clean(row.get("genre"))
    overview = clean(row.get("overview"))
    crew     = clean(row.get("crew"))
    lang     = clean(row.get("orig_lang"))
    budget   = clean(row.get("budget_x"))
    rev      = clean(row.get("revenue"))
    country  = clean(row.get("country"))

    text  = f"Title: {name}. Released in {date}, this {country} film is primarily in {lang}. "
    text += f"It falls under the {genres} genre. "
    text += f"It holds a rating of {score}/100. "
    text += f"Financially, it had a budget of ${budget} and made ${rev} in revenue. "
    text += f"Plot overview: {overview} "
    text += f"Key crew members include: {crew}."
    return text

df["content"] = df.apply(textify_movie, axis=1)
print("Sample content string:")
print(df["content"].iloc[0][:300], "...")

## 16 · Verify the cleaned dataset

In [ ]:
# final sanity checks
print(f"Final shape: {df.shape}")
print(f"Null check:\n{df.isnull().sum()}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 17 · Export (optional)

The cleaned dataframe is now ready for the embedding pipeline in `notebook.ipynb`.
Un-comment the cell below to save a CSV snapshot.

In [ ]:
# df.to_csv("../data/processed/cleaned_for_embeddings.csv", index=False)
# print("Saved!")